# Handwriting recognition: live demo
Loads the fine-tuned TrOCR model (no training) and reads handwriting from an uploaded photo or webcam.

**Before the demo:** Add Input -> Models -> Your Work -> **trocr-iam-words** -> Add. Turn Internet ON. GPU optional.
**Run All** (about 1-2 min), then open the public Gradio link printed by the last cell.

- **Single word** mode: best accuracy, this is what the model was trained on (IAM word crops).
- **Line / paragraph** mode: the image is split into lines and words automatically, and each word is read.

Tips for a photo: dark pen on white paper, write fairly large, good light, crop close to the writing.

In [1]:
   %pip install -q "transformers==4.57.1" sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ---- 1. Setup ----
import subprocess, sys, importlib, os, time
from pathlib import Path

def ensure(mod, pip_name=None):
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or mod])

for m, p in [("gradio", "gradio"), ("skimage", "scikit-image"), ("cv2", "opencv-python-headless")]:
    ensure(m, p)

import numpy as np, pandas as pd, cv2, torch
from PIL import Image, ImageDraw
from skimage.filters import threshold_sauvola
from transformers import VisionEncoderDecoderModel, TrOCRProcessor, AutoImageProcessor, RobertaTokenizer
import gradio as gr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| gradio", gr.__version__)

Device: cpu | gradio 6.26.0


In [3]:
# ---- 2. Find and load the saved model ----
MODEL_DIR = None     # leave None to find it automatically; or paste the path copied from the sidebar
HF_REPO = None       # or a Hugging Face repo id, e.g. "your-username/trocr-iam-words"
NUM_BEAMS = 2        # beam width chosen on validation in the main notebook

MODEL_NAME = "trocr-iam-words"   # name of your Kaggle Model (kaggle.com/models/athuljohnn/trocr-iam-words)

def _version_key(p):
    """Sort key so the newest model version (highest number folder) wins."""
    nums = [int(part) for part in p.parts if part.isdigit()]
    if nums:
        return nums[-1]
    return -1

def find_model_dir():
    if MODEL_DIR:
        return MODEL_DIR
    found = []
    root = Path("/kaggle/input")
    if root.exists():
        for f in root.rglob("model.safetensors"):
            d = f.parent
            if (d / "config.json").exists():
                found.append(d)
    if found:
        print("Models found under /kaggle/input:")
        for d in found:
            print("  ", d)
        mine = [d for d in found if MODEL_NAME in str(d)]
        if mine:
            return str(sorted(mine, key=_version_key)[-1])
        return str(sorted(found, key=_version_key)[-1])
    local = Path("/kaggle/working/results_iam/trocr_finetuned")     # same session as training
    if (local / "model.safetensors").exists():
        return str(local)
    if HF_REPO:
        return HF_REPO
    raise FileNotFoundError("No saved model found. In the right sidebar: Add Input -> Models -> Your Work -> "
                            f"{MODEL_NAME} -> Add. Or set MODEL_DIR to the folder path.")

SRC = find_model_dir()
print("Loading model from:", SRC)
t0 = time.time()
processor = TrOCRProcessor(image_processor=AutoImageProcessor.from_pretrained(SRC, use_fast=False),
                           tokenizer=RobertaTokenizer.from_pretrained(SRC))
model = VisionEncoderDecoderModel.from_pretrained(SRC).to(device).eval()
if device.type == "cuda":
    model = model.half()
print(f"Model ready in {time.time()-t0:.0f} s")

Models found under /kaggle/input:
   /kaggle/input/models/athuljohnn/trocr-iam-words/transformers/default/1
Loading model from: /kaggle/input/models/athuljohnn/trocr-iam-words/transformers/default/1
Model ready in 1 s


In [4]:
# ---- 3. Pre-processing (same 'binarized' pipeline the model was trained with) ----
def _odd(n):
    n = int(n)
    if n % 2:
        return n
    return n + 1

def ensure_dark_on_light(g):
    if g.mean() < 127:
        return 255 - g
    return g

def estimate_skew(g, max_angle=3.0, step=0.5):
    h, w = g.shape
    s = 400.0 / max(h, w)
    small = cv2.resize(g, (max(8, int(w * s)), max(8, int(h * s))), interpolation=cv2.INTER_AREA)
    ink = cv2.threshold(small, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1].astype(np.float32)
    hh, ww = ink.shape
    best, best_a = -1, 0.0
    for a in np.arange(-max_angle, max_angle + 1e-6, step):
        r = cv2.warpAffine(ink, cv2.getRotationMatrix2D((ww / 2, hh / 2), a, 1.0), (ww, hh))
        v = np.var(r.sum(1))
        if v > best:
            best, best_a = v, a
    return best_a

def deskew(g):
    a = estimate_skew(g)
    if abs(a) < 0.25:
        return g
    h, w = g.shape
    return cv2.warpAffine(g, cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0), (w, h),
                          flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def resize_height(g, h=128):
    s = h / g.shape[0]
    interp = cv2.INTER_CUBIC
    if s < 1:
        interp = cv2.INTER_AREA
    g = cv2.resize(g, (max(8, int(round(g.shape[1] * s))), h), interpolation=interp)
    return cv2.copyMakeBorder(g, 6, 6, 12, 12, cv2.BORDER_CONSTANT, value=255)

def preprocess_word(g):
    """Grayscale word crop (uint8) -> RGB PIL image in the model's training format."""
    g = deskew(ensure_dark_on_light(g))
    th = threshold_sauvola(g, window_size=_odd(max(15, g.shape[0] // 4)), k=0.2)
    g = ((g > th) * 255).astype(np.uint8)
    return Image.fromarray(resize_height(g)).convert("RGB")

In [5]:
# ---- 4. Splitting a photo into lines and words ----
def to_gray(img, max_side=1600):
    g = np.array(img.convert("L"))
    s = max_side / max(g.shape)
    if s < 1:
        g = cv2.resize(g, (int(g.shape[1] * s), int(g.shape[0] * s)), interpolation=cv2.INTER_AREA)
    return ensure_dark_on_light(g)

def ink_mask(g):
    """Binary mask of ink pixels; adaptive threshold copes with uneven phone lighting."""
    block = _odd(max(15, min(g.shape) // 15))
    m = cv2.adaptiveThreshold(cv2.GaussianBlur(g, (3, 3), 0), 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                              cv2.THRESH_BINARY_INV, block, 15)
    return cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))   # remove speckle

def ink_box(m, pad=6):
    ys, xs = np.nonzero(m)
    if len(xs) == 0:
        return None
    H, W = m.shape
    return (max(0, xs.min() - pad), max(0, ys.min() - pad), min(W, xs.max() + pad + 1), min(H, ys.max() + pad + 1))

def split_lines(m):
    """Row ranges that contain ink (horizontal projection profile)."""
    rows = (m > 0).sum(1)
    on = rows > max(2, 0.01 * m.shape[1])
    lines, start = [], None
    for y, v in enumerate(on):
        if v and start is None:
            start = y
        elif not v and start is not None:
            lines.append([start, y]); start = None
    if start is not None:
        lines.append([start, len(on)])
    merged = []
    for a, b in lines:                                  # join pieces split by small gaps (e.g. dots, descenders)
        if merged and a - merged[-1][1] < 8:
            merged[-1][1] = b
        else:
            merged.append([a, b])
    return [(a, b) for a, b in merged if b - a >= 10]

def _pieces(m_line, line_h):
    """Ink pieces in a line (letters / letter groups), with i-dots and accents merged into their letter."""
    k = max(2, int(line_h * 0.06))
    smeared = cv2.dilate(m_line, np.ones((max(3, line_h // 2), k), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(smeared, 8)
    boxes = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if w * h < 0.01 * line_h * line_h:              # ignore specks
            continue
        boxes.append([x, y, x + w, y + h])
    merged = []
    for b in sorted(boxes):
        overlap = 0
        if merged:
            overlap = min(b[2], merged[-1][2]) - b[0]
        if merged and overlap > 0.5 * min(b[2] - b[0], merged[-1][2] - merged[-1][0]):
            m = merged[-1]
            m[0], m[1], m[2], m[3] = min(m[0], b[0]), min(m[1], b[1]), max(m[2], b[2]), max(m[3], b[3])
        else:
            merged.append(b)
    return merged

def _gap_threshold(gaps, line_h):
    """Split gaps into 'between letters' and 'between words' (largest jump in the sorted gaps)."""
    floor = 0.12 * line_h                               # never split narrower than this
    big = [g for g in gaps if g > floor]
    if not big:
        return None
    s = sorted(gaps)
    best_jump, thr = 0.0, None
    for a, b in zip(s, s[1:]):
        if b > floor and b - a > best_jump:
            best_jump, thr = b - a, (a + b) / 2
    if thr is None or best_jump < 0.06 * line_h:        # gaps all similar: treat all wide ones as word gaps
        thr = floor
    return max(thr, floor)

def split_words(m_line, line_h):
    """Word boxes in one line: group ink pieces, starting a new word at clearly wider gaps."""
    pieces = _pieces(m_line, line_h)
    if not pieces:
        return []
    gaps = [b[0] - a[2] for a, b in zip(pieces, pieces[1:])]
    thr = _gap_threshold(gaps, line_h)
    words = [list(pieces[0])]
    for g, p in zip(gaps, pieces[1:]):
        if thr is not None and g > thr:
            words.append(list(p))
        else:
            w = words[-1]
            w[0], w[1], w[2], w[3] = min(w[0], p[0]), min(w[1], p[1]), max(w[2], p[2]), max(w[3], p[3])
    return [tuple(w) for w in words]

def segment(img, mode):
    """Returns (gray image, list of lines, each a list of (x0, y0, x1, y1) word boxes)."""
    g = to_gray(img)
    m = ink_mask(g)
    if mode == "Single word":
        b = ink_box(m)
        if b is None:
            return g, []
        return g, [[b]]
    out = []
    for a, b in split_lines(m):
        words = []
        for x0, y0, x1, y1 in split_words(m[a:b], b - a):
            pad = 4
            words.append((max(0, x0 - pad), max(0, a + y0 - pad), min(g.shape[1], x1 + pad), min(g.shape[0], a + y1 + pad)))
        if words:
            out.append(words)
    return g, out

In [6]:
# ---- 5. Recognition ----
@torch.no_grad()
def read_words(crops, batch=16):
    texts, confs = [], []
    for i in range(0, len(crops), batch):
        pv = processor(images=crops[i:i + batch], return_tensors="pt").pixel_values.to(device)
        if device.type == "cuda":
            pv = pv.half()
        out = model.generate(pv, num_beams=NUM_BEAMS, max_length=32, early_stopping=True, no_repeat_ngram_size=0,
                             length_penalty=1.0, output_scores=True, return_dict_in_generate=True)
        texts += [" ".join(t.split()) for t in processor.tokenizer.batch_decode(out.sequences, skip_special_tokens=True)]
        if NUM_BEAMS > 1:
            confs += torch.exp(out.sequences_scores.float()).cpu().tolist()
        else:
            confs += [float("nan")] * pv.shape[0]
    return texts, confs

def recognise(img, mode="Single word"):
    if img is None:
        return "", None, None
    t0 = time.time()
    g, lines = segment(img, mode)
    boxes = [b for line in lines for b in line]
    if not boxes:
        return "(no handwriting found)", img, None
    crops = [preprocess_word(g[y0:y1, x0:x1]) for x0, y0, x1, y1 in boxes]
    texts, confs = read_words(crops)
    # rebuild text line by line
    k, lines_txt = 0, []
    for line in lines:
        lines_txt.append(" ".join(texts[k:k + len(line)])); k += len(line)
    # draw boxes on the image
    vis = Image.fromarray(g).convert("RGB"); d = ImageDraw.Draw(vis)
    for (x0, y0, x1, y1), t in zip(boxes, texts):
        d.rectangle([x0, y0, x1, y1], outline=(220, 40, 40), width=2)
        d.text((x0, max(0, y0 - 12)), t, fill=(220, 40, 40))
    table = pd.DataFrame({"word": texts, "confidence": [round(c, 3) for c in confs]})
    print(f"{len(boxes)} word(s) read in {time.time()-t0:.1f} s")
    return "\n".join(lines_txt), vis, table

In [7]:
# ---- 6. Quick self-check on real held-out IAM words (if the IAM dataset is attached) ----
manifest = Path("/kaggle/working/results_iam/iam_split_manifest.csv")
if manifest.exists():
    test = pd.read_csv(manifest)
    test = test[test.split == "test"].sample(8, random_state=1)
    preds, _ = read_words([preprocess_word(to_gray(Image.open(p))) for p in test.path])
    display(pd.DataFrame({"correct": test.text.tolist(), "model": preds}))
else:
    print("No split manifest in this session - skipping the self-check (the demo below still works).")

No split manifest in this session - skipping the self-check (the demo below still works).


In [8]:
# ---- 7. Launch the demo app ----
import inspect
extra = {}
params = inspect.signature(gr.Interface).parameters
if "flagging_mode" in params:
    extra["flagging_mode"] = "never"
elif "allow_flagging" in params:
    extra["allow_flagging"] = "never"
demo = gr.Interface(
    fn=recognise,
    inputs=[gr.Image(type="pil", sources=["upload", "webcam", "clipboard"], label="Handwriting image"),
            gr.Radio(["Single word", "Line / paragraph"], value="Single word", label="Mode")],
    outputs=[gr.Textbox(label="Recognised text", lines=4),
             gr.Image(label="What the model looked at"),
             gr.Dataframe(label="Per-word confidence")],
    title="Handwriting recognition (TrOCR fine-tuned on IAM)",
    description="Upload or take a photo of handwriting. Single word mode is most accurate.",
    **extra,
)
demo.launch(share=True)   # prints a public link that works in any browser

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2e07447b1905962031.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1 word(s) read in 9.4 s
